# Solar forecast evaluation and blended forecast

## Purpose

We use this notebook to build a transparent chain from measurement quality control to a blended forecast. The first part checks one year of DMI GHI measurements. The second part compares the available forecast models by lead time. The last part selects a simple blend using separate development and validation periods, checks it on a later holdout period, and prepares a competition forecast.

Our main project question is: **which available models provide useful GHI forecasts at lead times from 10 to 105 hours, and does a simple lead-time-dependent blend improve on the best single model?**

We treat the selected cloudy, variable, and sunny days as illustrations. The statistical comparison across the full forecast archive is the main evidence for choosing the blend.


## Notebook flow

1. Inspect raw measurements and data coverage.
2. Check BSRN limits, possible shading, and possible thermal offset.
3. Resample measurements to hourly values and identify daylight.
4. Match forecast values to measurements and compare models by lead time.
5. Select a simple candidate blend using development data; choose between it and the best single model using validation data.
6. Report the selected strategy on a later holdout period and inspect three daily profiles.
7. On the submission date, create a forecast from the specified forecast issue time.

The forecast competition's exact scoring rule and timestamps should take precedence over our analysis defaults. We keep all times in UTC.


In [ ]:
from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pvlib
import solarpy

# Find the project root whether Jupyter was started in the repository
# root or in its notebooks folder.
project_root = Path.cwd().resolve()
if not (project_root / "scripts" / "functions.py").exists():
    project_root = project_root.parent
if not (project_root / "scripts" / "functions.py").exists():
    raise FileNotFoundError(
        "Could not find scripts/functions.py. Start Jupyter from the project "
        "folder or its notebooks folder."
    )

sys.path.insert(0, str(project_root))
import scripts.functions as functions
importlib.reload(functions)

from scripts.functions import (
    get_dmi_ghi,
    resample_dmi_ghi_hourly,
    get_forecast_data,
    our_blend_by_lead_time,
)


In [ ]:
# Study configuration. Update these dates explicitly when the archive changes.
STATION = "06019"
LATITUDE = 56.93
LONGITUDE = 8.6412

# One year of measurements ending on the latest complete day in this study.
MEASUREMENT_START = "2025-11-24"
MEASUREMENT_END = "2026-11-24"

# Forecast archive available for the model comparison.
FORECAST_START = "2026-11-09"
FORECAST_END = "2026-11-24 23:00"

LEAD_INTERVALS = [(10, 33), (34, 57), (58, 81), (82, 105)]
MIN_HOURLY_OBSERVATIONS = 6
MIN_PAIRS_TO_DISPLAY = 30

PROFILE_DAYS = {
    "2026-11-16": "Overcast candidate",
    "2026-11-17": "Variable conditions candidate",
    "2026-11-21": "Sunny candidate",
}
PROFILE_LEAD_H = 24

FORECAST_MODELS = [
    "dmi_harmonie_arome_europe",
    "metno_nordic",
    "ecmwf_ifs025",
    "ecmwf_aifs025_single",
    "gfs_global",
    "ecmwf_ifs",
    "dmi_seamless",
    "gfs_seamless",
]


# 1. Raw measurement data

We start with the full 10-minute DMI record. Before calculating forecast errors, we want to understand whether the measurement series has gaps, duplicates, non-finite values, or physically implausible observations. Quality flags are diagnostics; we do not automatically delete flagged observations without inspecting them.


In [ ]:
data_raw = get_dmi_ghi(
    start=MEASUREMENT_START,
    end=MEASUREMENT_END,
    station=STATION,
)

data_raw = data_raw.sort_index()
data_raw.info()
display(data_raw.head())
display(data_raw["ghi"].describe())


In [ ]:
ghi_raw = pd.to_numeric(data_raw["ghi"], errors="coerce")
time_steps = data_raw.index.to_series().diff().dropna()

raw_quality = pd.Series({
    "number of rows": len(data_raw),
    "missing or non-numeric GHI": int(ghi_raw.isna().sum()),
    "infinite GHI values": int(np.isinf(ghi_raw).sum()),
    "negative GHI values": int((ghi_raw < 0).sum()),
    "duplicate timestamps": int(data_raw.index.duplicated().sum()),
    "gaps longer than 10 minutes": int((time_steps > pd.Timedelta(minutes=10)).sum()),
    "largest timestamp gap": time_steps.max(),
}, name="raw measurement check")

display(raw_quality.to_frame())


### Initial data-quality interpretation

A gap is not automatically a bad measurement; it means there are fewer observations available for the affected hour. Negative or non-finite values need attention before resampling. We retain the observation count at hourly resolution so incomplete hours can be excluded from forecast verification transparently.


# 2. Solar position and BSRN quality limits

The DMI measurements represent 10-minute intervals. We use the interval midpoint for solar position, consistent with the measurement notebook. We then compare measured GHI with the BSRN physical (PPL) and extremely rare (ERL) upper limits. These limits help identify suspicious points; an ERL flag is not by itself proof that a measurement is wrong.


In [ ]:
measurement_midpoints = data_raw.index - pd.Timedelta(minutes=5)

solpos_raw = pvlib.solarposition.get_solarposition(
    measurement_midpoints,
    latitude=LATITUDE,
    longitude=LONGITUDE,
    method="nrel_numpy",
)

data_raw["solar_zenith"] = solpos_raw["zenith"].to_numpy()
data_raw["solar_elevation"] = solpos_raw["apparent_elevation"].to_numpy()
data_raw["solar_azimuth"] = solpos_raw["azimuth"].to_numpy()
data_raw["dni_extra"] = pvlib.irradiance.get_extra_radiation(
    measurement_midpoints
).to_numpy()
data_raw["ghi_extra"] = data_raw["dni_extra"] * np.maximum(
    np.cos(np.radians(data_raw["solar_zenith"])), 0
)

data_raw["ghi_ppl_flag"] = solarpy.quality.bsrn_limits_flag(
    data_raw["ghi"], data_raw["solar_zenith"], data_raw["dni_extra"],
    limits="ghi-ppl", check="upper",
)
data_raw["ghi_erl_flag"] = solarpy.quality.bsrn_limits_flag(
    data_raw["ghi"], data_raw["solar_zenith"], data_raw["dni_extra"],
    limits="ghi-erl", check="upper",
)

_, data_raw["ghi_ppl_upper"] = solarpy.quality.bsrn_limits(
    data_raw["solar_zenith"], data_raw["dni_extra"], limits="ghi-ppl"
)
_, data_raw["ghi_erl_upper"] = solarpy.quality.bsrn_limits(
    data_raw["solar_zenith"], data_raw["dni_extra"], limits="ghi-erl"
)


In [ ]:
fig, ax = solarpy.plotting.plot_bsrn_limits(
    irradiance=data_raw["ghi"],
    component="ghi",
    ghi_extra=data_raw["ghi_extra"],
)
fig.set_size_inches(11, 6)
ax.set_title("BSRN quality-control limits for measured GHI")
ax.set_xlabel("Extraterrestrial horizontal irradiance [W/m²]")
ax.set_ylabel("Measured GHI [W/m²]")
ax.set_xlim(0, 1200)
ax.set_ylim(0, 1200)
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
ppl_flag = data_raw["ghi_ppl_flag"].fillna(False).astype(bool)
erl_flag = data_raw["ghi_erl_flag"].fillna(False).astype(bool)

flag_summary = pd.Series({
    "above PPL upper limit": int(ppl_flag.sum()),
    "above ERL upper limit": int(erl_flag.sum()),
    "ERL only (above ERL but not PPL)": int((erl_flag & ~ppl_flag).sum()),
}, name="flagged observations")

display(flag_summary.to_frame())

flagged_examples = data_raw.loc[
    ppl_flag | erl_flag,
    ["ghi", "ghi_erl_upper", "ghi_ppl_upper", "solar_zenith"],
]
display(flagged_examples.head(20))


In [ ]:
# Inspect the first flagged day, if any flags were found.
if not flagged_examples.empty:
    flagged_day = flagged_examples.index[0].date()
    day_raw = data_raw.loc[str(flagged_day)]

    fig, ax = plt.subplots(figsize=(10, 4))
    day_raw["ghi"].plot(ax=ax, label="Measured GHI")
    ax.plot(day_raw.index, day_raw["ghi_erl_upper"], label="ERL upper limit")
    ax.plot(day_raw.index, day_raw["ghi_ppl_upper"], label="PPL upper limit")
    ax.set_title(f"Inspecting a flagged measurement day: {flagged_day}")
    ax.set_xlabel("Time [UTC]")
    ax.set_ylabel("GHI [W/m²]")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No observations exceeded the selected BSRN upper limits.")


## Possible shading pattern

The next plot bins irradiance by solar azimuth and elevation. A persistent low-irradiance region in a particular direction can indicate local shading or horizon obstruction. It is a diagnostic, not a correction: we do not alter GHI values based on this plot.

Here the shading indicator uses the clearness index relative to extraterrestrial horizontal irradiance, (K_t = GHI / GHI_{extra}). This is different from the clear-sky index used later to describe cloud conditions.


In [ ]:
valid_shading = (
    (data_raw["solar_elevation"] > 3)
    & data_raw["ghi"].notna()
    & (data_raw["ghi"] >= 0)
    & (data_raw["ghi_extra"] > 50)
)

fig, ax = plt.subplots(figsize=(10, 4.5))
solarpy.plotting.plot_shading_heatmap(
    value=data_raw.loc[valid_shading, "ghi"] / data_raw.loc[valid_shading, "ghi_extra"],
    solar_azimuth=data_raw.loc[valid_shading, "solar_azimuth"],
    solar_elevation=data_raw.loc[valid_shading, "solar_elevation"],
    azimuth_bin_size=5,
    elevation_bin_size=2,
    encoding="max",
    cmap="viridis",
    colorbar_label="Kt = GHI / extraterrestrial horizontal GHI [-]",
    ax=ax,
)
ax.set_title("Directional irradiance pattern: possible shading")
fig.tight_layout()
plt.show()


## Possible thermal offset

We inspect GHI through the day across the year and overlay calculated sunrise and sunset. A repeatable offset of the measured daily profile relative to sunrise or sunset can be a clue to sensor or timestamp effects. We treat the plot as a prompt for investigation rather than a standalone diagnosis.


In [ ]:
cmap, norm = solarpy.plotting.irradiance_colormap_and_norm(vmax=1000)
fig, ax = plt.subplots(figsize=(13, 5))

solarpy.plotting.plot_intraday_heatmap(
    time=measurement_midpoints.tz_localize(None),
    values=data_raw["ghi"].to_numpy(),
    time_resolution=10,
    cmap=cmap,
    norm=norm,
    colorbar_label="GHI [W/m²]",
    ax=ax,
)

days = pd.date_range(
    measurement_midpoints.min().normalize(),
    measurement_midpoints.max().normalize(),
    freq="D",
)
sun = pvlib.solarposition.sun_rise_set_transit_spa(
    days, latitude=LATITUDE, longitude=LONGITUDE
)

for column, label in [("sunrise", "Sunrise"), ("sunset", "Sunset")]:
    hour_utc = (sun[column] - sun.index.normalize()).dt.total_seconds() / 3600
    ax.plot(days.tz_localize(None), hour_utc, "r--", linewidth=1, label=label)

ax.set_title("Measured GHI by date and time, with sunrise and sunset")
ax.set_xlabel("Date")
ax.set_ylabel("Time of day [UTC]")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()


# 3. Hourly measurements and daylight

The forecasts are hourly, so we average the 10-minute measurements into right-labeled hourly intervals. We require all six 10-minute observations for a complete hour. We keep the observation count so incomplete hours remain visible in the quality summary.

For the model-skill analysis, we remove hours when the sun is below the horizon. This avoids letting the many near-zero night values dominate the daylight comparison. The final competition file will still contain every required timestamp; we will follow the competition's scoring rule when choosing the final objective.


In [ ]:
hourly_dmi = resample_dmi_ghi_hourly(
    data_raw[["ghi"]],
    min_observations=MIN_HOURLY_OBSERVATIONS,
)

display(hourly_dmi.head())
display(hourly_dmi[["ghi_measured", "n_observations"]].describe())


In [ ]:
hourly_quality = pd.Series({
    "number of hourly bins": len(hourly_dmi),
    "complete hours (6 observations)": int(
        (hourly_dmi["n_observations"] == 6).sum()
    ),
    "incomplete or empty hours": int(
        (hourly_dmi["n_observations"] < MIN_HOURLY_OBSERVATIONS).sum()
    ),
    "missing hourly GHI": int(hourly_dmi["ghi_measured"].isna().sum()),
}, name="hourly measurement check")
display(hourly_quality.to_frame())


In [ ]:
fig, ax = solarpy.plotting.plot_intraday_heatmap(
    time=hourly_dmi.index,
    values=hourly_dmi["ghi_measured"],
    time_resolution=60,
    cmap="plasma",
    colorbar_label="Hourly mean GHI [W/m²]",
)
fig.set_size_inches(12, 5)
ax.set_title("Hourly measured GHI by date and UTC hour")
ax.set_xlabel("Date")
ax.set_ylabel("Hour of day [UTC]")
plt.tight_layout()
plt.show()


In [ ]:
hourly_solpos = pvlib.solarposition.get_solarposition(
    hourly_dmi.index,
    latitude=LATITUDE,
    longitude=LONGITUDE,
    method="nrel_numpy",
)
hourly_dmi["solar_elevation"] = hourly_solpos["apparent_elevation"].to_numpy()

# Keep the full hourly table for matching. Use this daylight subset for
# daylight forecast-skill metrics.
daylight_dmi = hourly_dmi.loc[
    (hourly_dmi["solar_elevation"] > 0)
    & hourly_dmi["ghi_measured"].notna()
].copy()

print(f"Complete daylight hours available: {len(daylight_dmi):,}")


# 4. Clear-sky index for describing daily profiles

For cloud conditions, we use the **clear-sky index**, (K_c = GHI_{measured} / GHI_{clear}), where clear-sky GHI is estimated for this location and timestamp. This normalizes the seasonal and daily solar geometry better than using measured GHI alone. We exclude very low clear-sky irradiance from the ratio because division by a small value is unstable. The cutoff below is a stated analysis choice; we should check that the broad interpretation of the selected days is not driven by it.

The clear-sky model below uses a site altitude of 0 m because station elevation was not included in the project metadata; update it if the station elevation is available. The three dates below are our initial labels. The clear-sky-index summary lets us check whether those labels are plausible. These are retrospective descriptions based on measured weather, not information that would have been available when issuing a forecast.


In [ ]:
location = pvlib.location.Location(
    latitude=LATITUDE,
    longitude=LONGITUDE,
    tz="UTC",
    altitude=0,
    name=f"DMI station {STATION}",
)
clear_sky = location.get_clearsky(hourly_dmi.index, model="ineichen")
hourly_dmi["ghi_clear_sky"] = clear_sky["ghi"].to_numpy()

# Avoid unstable ratios close to sunrise and sunset.
KC_MIN_SOLAR_ELEVATION = 5
KC_MIN_CLEAR_GHI = 50
kc_valid = (
    (hourly_dmi["solar_elevation"] > KC_MIN_SOLAR_ELEVATION)
    & (hourly_dmi["ghi_clear_sky"] >= KC_MIN_CLEAR_GHI)
    & hourly_dmi["ghi_measured"].notna()
    & (hourly_dmi["ghi_measured"] >= 0)
)
hourly_dmi["kc"] = np.nan
hourly_dmi.loc[kc_valid, "kc"] = (
    hourly_dmi.loc[kc_valid, "ghi_measured"]
    / hourly_dmi.loc[kc_valid, "ghi_clear_sky"]
)


In [ ]:
profile_summary = []
for date, label in PROFILE_DAYS.items():
    start = pd.Timestamp(date, tz="UTC")
    end = start + pd.Timedelta(days=1)
    day = hourly_dmi.loc[(hourly_dmi.index > start) & (hourly_dmi.index <= end)]
    kc = day["kc"].dropna()
    profile_summary.append({
        "date": date,
        "initial description": label,
        "hours used for Kc": len(kc),
        "median Kc": kc.median(),
        "mean Kc": kc.mean(),
        "within-day Kc standard deviation": kc.std(),
    })

profile_summary = pd.DataFrame(profile_summary)
display(profile_summary.round(3))


# 5. Forecast data and matched pairs

We now shift from measurement quality to forecast verification. Each forecast is identified by its issue time, valid time, and lead time. The valid time is the timestamp used to match the forecast to the hourly measured GHI.

We first check model availability and coverage. Different models may have different missing timestamps or forecast horizons, so we keep sample counts beside every metric. For the main model comparison we use daylight valid times.


In [ ]:
forecast_data = get_forecast_data(
    hourly_dmi,
    station=STATION,
    start=FORECAST_START,
    end=FORECAST_END,
    models=FORECAST_MODELS,
)

forecast_coverage = []
for model, frame in forecast_data.items():
    forecast_coverage.append({
        "model": model,
        "forecast rows": len(frame),
        "matched measurements": int(frame["ghi_measured"].notna().sum()),
        "first issue time": frame["issue_time"].min() if len(frame) else pd.NaT,
        "last issue time": frame["issue_time"].max() if len(frame) else pd.NaT,
        "maximum lead [h]": frame["lead_time_h"].max() if len(frame) else np.nan,
    })

display(pd.DataFrame(forecast_coverage).set_index("model"))


In [ ]:
daylight_times = daylight_dmi.index
forecast_data_daylight = {
    model: frame.loc[frame["valid_time"].isin(daylight_times)].copy()
    for model, frame in forecast_data.items()
}

# Pivot each model to valid_time x lead_time for clear comparisons and blending.
forecast_unstacked = {
    model: frame.set_index(["valid_time", "lead_time_h"])["ghi_forecast"]
    .unstack()
    .sort_index()
    for model, frame in forecast_data_daylight.items()
}

for model, frame in forecast_unstacked.items():
    print(
        model,
        "| valid times:", len(frame),
        "| lead-time columns:", len(frame.columns),
        "| maximum lead [h]:", frame.columns.max() if len(frame.columns) else np.nan,
    )


## Metric convention

For each model and lead time, we calculate RMSE, MAE, and bias. We use the course convention **bias = measured minus forecast**. Positive bias means the forecast tends to underestimate measured GHI; negative bias means it tends to overestimate.

The score function below is written in this notebook so the bias convention is explicit and does not depend on an older local copy of the helper function.


In [ ]:
def calculate_metrics_by_lead(forecasts, measurements):
    # Return RMSE, bias, MAE, and pair count for every model and lead time.
    results = {}
    measured = measurements["ghi_measured"]

    for model, forecast in forecasts.items():
        forecast = forecast.replace([np.inf, -np.inf], np.nan)
        observations = measured.reindex(forecast.index).replace(
            [np.inf, -np.inf], np.nan
        )
        errors = forecast.rsub(observations, axis=0)

        scores = pd.DataFrame({
            "rmse": errors.pow(2).mean().pow(0.5),
            "bias": errors.mean(),
            "mae": errors.abs().mean(),
            "n": errors.count(),
        })
        scores.index.name = "lead_time_h"
        results[model] = scores.loc[scores.index >= 1].sort_index()

    return pd.concat(results, names=["model"]).reset_index()


daylight_scores = calculate_metrics_by_lead(
    forecast_unstacked,
    daylight_dmi,
)


In [ ]:
# Focus on the lead times relevant to the project question.
project_scores = daylight_scores.loc[
    daylight_scores["lead_time_h"].between(10, 105)
].copy()

# Hide unstable points with very few pairs in the plot, but retain them in the table.
plot_scores = project_scores.copy()
plot_scores.loc[
    plot_scores["n"] < MIN_PAIRS_TO_DISPLAY,
    ["rmse", "bias", "mae"],
] = np.nan

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True, constrained_layout=True)
for ax, metric in zip(axes, ["rmse", "bias", "mae"]):
    for model, group in plot_scores.groupby("model", sort=False):
        group = group.sort_values("lead_time_h")
        ax.plot(group["lead_time_h"], group[metric], label=model, linewidth=1.5)
    ax.set_title(metric.upper())
    ax.set_ylabel("GHI error [W/m²]")
    ax.grid(alpha=0.3)
    if metric == "bias":
        ax.axhline(0, color="black", linestyle="--", linewidth=1)
axes[-1].set_xlabel("Forecast lead time [hours]")
axes[0].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.suptitle("Daylight forecast errors by lead time")
plt.show()


### What we learn from the lead-time curves

We use these curves to identify patterns, not to declare one universal winner. We should ask: which models have the lowest RMSE at each horizon, does bias grow or change sign with lead time, and where is the sample size too small to support a comparison?


In [ ]:
def summarize_lead_intervals(scores, intervals, mean_daylight_ghi):
    rows = []
    for first, last in intervals:
        for model, group in scores.loc[
            scores["lead_time_h"].between(first, last)
        ].groupby("model"):
            n = group["n"].sum()
            if n == 0:
                continue
            rmse = np.sqrt((group["rmse"].pow(2) * group["n"]).sum() / n)
            bias = (group["bias"] * group["n"]).sum() / n
            mae = (group["mae"] * group["n"]).sum() / n
            rows.append({
                "interval": f"{first}-{last} h",
                "model": model,
                "rmse": rmse,
                "nRMSE [% of mean daylight GHI]": (
                    100 * rmse / mean_daylight_ghi
                    if mean_daylight_ghi > 0 else np.nan
                ),
                "mae": mae,
                "bias": bias,
                "n forecast pairs": n,
            })
    return pd.DataFrame(rows)

mean_daylight_ghi = daylight_dmi["ghi_measured"].mean()
interval_scores = summarize_lead_intervals(
    project_scores, LEAD_INTERVALS, mean_daylight_ghi
)
display(interval_scores.sort_values(["interval", "rmse"]).round(2))


In [ ]:
# Compare models across the four project lead-time intervals.
metrics_to_plot = [
    ("nRMSE [% of mean daylight GHI]", "Normalized RMSE", "%"),
    ("mae", "MAE", "W/m²"),
    ("bias", "Bias (measured - forecast)", "W/m²"),
]
fig, axes = plt.subplots(
    len(LEAD_INTERVALS), len(metrics_to_plot),
    figsize=(16, 13), constrained_layout=True,
)

for row, (first, last) in enumerate(LEAD_INTERVALS):
    label = f"{first}-{last} h"
    frame = interval_scores.loc[
        interval_scores["interval"] == label
    ].sort_values("rmse")
    for col, (metric, title, unit) in enumerate(metrics_to_plot):
        ax = axes[row, col]
        ax.set_title(f"{label}: {title}")
        ax.set_xlabel(unit)
        if frame.empty:
            ax.text(0.5, 0.5, "No matched data", ha="center", va="center", transform=ax.transAxes)
            ax.set_yticks([])
            continue
        y = np.arange(len(frame))
        ax.barh(y, frame[metric], color=plt.get_cmap("tab10")(row))
        ax.set_yticks(y)
        ax.set_yticklabels(frame["model"] if col == 0 else [""] * len(frame))
        ax.invert_yaxis()
        ax.grid(axis="x", alpha=0.3)
        ax.set_axisbelow(True)
        if metric == "bias":
            ax.axvline(0, color="black", linewidth=1)
fig.suptitle("Daylight model performance by lead-time interval")
plt.show()


## A note on fair comparisons

The score curves use every available matched pair for each model, so sample sizes can differ. We show `n` to make this visible. When we directly compare a candidate blend with its best single model, we will use only the exact forecast–measurement pairs available for both. This paired comparison is the one we use to decide whether the blend is worth keeping.


# 6. Select a simple blend without using the final holdout

We split the historical valid times into three chronological periods. All lead times for the same valid time stay in the same period. This avoids randomly placing different forecasts for the same observed hour in both development and evaluation data.

- **Development:** 9–15 November. Screen models separately in each lead interval.
- **Validation:** 16–19 November. Compare the equal-weight blend of the two development leaders with the best single model, on common pairs.
- **Holdout:** 20–24 November. Evaluate the chosen strategy once, without changing the weights based on these results.

These are short periods, so results are exploratory. A more robust study would repeat the exercise over several longer date blocks or seasons.


In [ ]:
DEVELOPMENT_START = "2026-11-09"
DEVELOPMENT_END = "2026-11-15 23:00"
VALIDATION_START = "2026-11-16"
VALIDATION_END = "2026-11-19 23:00"
HOLDOUT_START = "2026-11-20"
HOLDOUT_END = "2026-11-24 23:00"


def utc_timestamp(value):
    value = pd.Timestamp(value)
    return value.tz_localize("UTC") if value.tzinfo is None else value.tz_convert("UTC")


def restrict_to_period(forecasts, start, end):
    start, end = utc_timestamp(start), utc_timestamp(end)
    return {
        model: frame.loc[(frame.index >= start) & (frame.index <= end)]
        for model, frame in forecasts.items()
    }


def period_interval_summary(forecasts, measurements, start, end):
    period_forecasts = restrict_to_period(forecasts, start, end)
    period_measurements = measurements.loc[
        (measurements.index >= utc_timestamp(start))
        & (measurements.index <= utc_timestamp(end))
    ]
    scores = calculate_metrics_by_lead(period_forecasts, period_measurements)
    scores = scores.loc[scores["lead_time_h"].between(10, 105)]
    return summarize_lead_intervals(scores, LEAD_INTERVALS, mean_daylight_ghi)


dev_summary = period_interval_summary(
    forecast_unstacked, daylight_dmi, DEVELOPMENT_START, DEVELOPMENT_END
)
validation_summary = period_interval_summary(
    forecast_unstacked, daylight_dmi, VALIDATION_START, VALIDATION_END
)
holdout_summary = period_interval_summary(
    forecast_unstacked, daylight_dmi, HOLDOUT_START, HOLDOUT_END
)

display(dev_summary.sort_values(["interval", "rmse"]).round(2))


In [ ]:
# Screen for the two lowest-RMSE individual models in each interval.
# Their validation comparison below uses identical paired cases.
top_models_by_interval = {}
for first, last in LEAD_INTERVALS:
    label = f"{first}-{last} h"
    ranked = (
        dev_summary.loc[dev_summary["interval"] == label]
        .dropna(subset=["rmse"])
        .sort_values("rmse")
    )
    top_models_by_interval[(first, last)] = ranked["model"].head(2).tolist()

display(pd.DataFrame([
    {
        "lead interval": f"{first}-{last} h",
        "development leaders": ", ".join(top_models_by_interval[(first, last)]),
    }
    for first, last in LEAD_INTERVALS
]))


### Candidate rule

For each interval, our simple candidate is an equal-weight blend of the two development leaders. We compare it against the better development model on validation data. Equal weights are a transparent baseline; we avoid fitting many free weights to a short archive. If the blend does not improve validation RMSE on common pairs, we keep the best single model for that interval.


In [ ]:
candidate_schedule = []
for first, last in LEAD_INTERVALS:
    leaders = top_models_by_interval[(first, last)]
    if len(leaders) >= 2:
        weights = {leaders[0]: 0.5, leaders[1]: 0.5}
    elif len(leaders) == 1:
        weights = {leaders[0]: 1.0}
    else:
        raise ValueError(f"No development data for lead interval {first}-{last} h")
    candidate_schedule.append((first, last, weights))

candidate_blend = our_blend_by_lead_time(
    forecast_unstacked,
    candidate_schedule,
)

display(pd.DataFrame([
    {"lead interval": f"{first}-{last} h", "candidate weights": weights}
    for first, last, weights in candidate_schedule
]))


In [ ]:
def compare_blend_to_interval_leader(
    blend, forecasts, measurements, start, end,
    intervals, leaders_by_interval,
):
    rows = []
    start, end = utc_timestamp(start), utc_timestamp(end)
    measured = measurements["ghi_measured"]

    for first, last in intervals:
        leaders = leaders_by_interval[(first, last)]
        if not leaders:
            continue
        leader = leaders[0]
        lead_columns = [
            lead for lead in blend.columns if first <= lead <= last
        ]

        blend_series = blend.loc[
            (blend.index >= start) & (blend.index <= end), lead_columns
        ].stack().rename("blend")
        leader_series = forecasts[leader].loc[
            (forecasts[leader].index >= start)
            & (forecasts[leader].index <= end)
        ].reindex(columns=lead_columns).stack().rename("leader")

        paired = pd.concat([blend_series, leader_series], axis=1).dropna()
        if paired.empty:
            continue

        observations = measured.reindex(paired.index.get_level_values(0))
        observations.index = paired.index
        valid = observations.notna()
        paired = paired.loc[valid]
        observations = observations.loc[valid]

        blend_error = observations - paired["blend"]
        leader_error = observations - paired["leader"]
        blend_rmse = np.sqrt(blend_error.pow(2).mean())
        leader_rmse = np.sqrt(leader_error.pow(2).mean())

        rows.append({
            "interval": f"{first}-{last} h",
            "development leader": leader,
            "blend RMSE": blend_rmse,
            "leader RMSE": leader_rmse,
            "common pairs": len(paired),
            "blend improves validation": blend_rmse < leader_rmse,
        })

    return pd.DataFrame(rows)


validation_comparison = compare_blend_to_interval_leader(
    candidate_blend,
    forecast_unstacked,
    daylight_dmi,
    VALIDATION_START,
    VALIDATION_END,
    LEAD_INTERVALS,
    top_models_by_interval,
)
display(validation_comparison.round(2))


In [ ]:
# Keep the equal-weight blend only in intervals where it improves validation RMSE.
final_weight_schedule = []
for first, last, candidate_weights in candidate_schedule:
    label = f"{first}-{last} h"
    result = validation_comparison.loc[
        validation_comparison["interval"] == label
    ]
    leader = top_models_by_interval[(first, last)][0]

    if not result.empty and bool(result.iloc[0]["blend improves validation"]):
        chosen_weights = candidate_weights
    else:
        chosen_weights = {leader: 1.0}

    final_weight_schedule.append((first, last, chosen_weights))

final_blend = our_blend_by_lead_time(
    forecast_unstacked,
    final_weight_schedule,
)

display(pd.DataFrame([
    {"lead interval": f"{first}-{last} h", "selected weights": weights}
    for first, last, weights in final_weight_schedule
]))


# 7. Holdout evaluation

We now evaluate the selected schedule on 20–24 November. This period was not used to rank models or decide whether to use a blend. We compare the final schedule with the development leader using identical forecast–measurement pairs. If the holdout result is worse, we report that honestly; we do not tune the schedule on this period and then reuse it as a test.


In [ ]:
holdout_comparison = compare_blend_to_interval_leader(
    final_blend,
    forecast_unstacked,
    daylight_dmi,
    HOLDOUT_START,
    HOLDOUT_END,
    LEAD_INTERVALS,
    top_models_by_interval,
)
display(holdout_comparison.round(2))


In [ ]:
# Lead-time curves for the selected blend and individual models on holdout data.
holdout_forecasts = restrict_to_period(
    {**forecast_unstacked, "selected blend": final_blend},
    HOLDOUT_START,
    HOLDOUT_END,
)
holdout_measurements = daylight_dmi.loc[
    (daylight_dmi.index >= utc_timestamp(HOLDOUT_START))
    & (daylight_dmi.index <= utc_timestamp(HOLDOUT_END))
]
holdout_scores = calculate_metrics_by_lead(
    holdout_forecasts, holdout_measurements
)
holdout_scores = holdout_scores.loc[
    holdout_scores["lead_time_h"].between(10, 105)
]

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True, constrained_layout=True)
for ax, metric in zip(axes, ["rmse", "bias", "mae"]):
    for model, group in holdout_scores.groupby("model", sort=False):
        group = group.sort_values("lead_time_h")
        is_blend = model == "selected blend"
        ax.plot(
            group["lead_time_h"], group[metric], label=model,
            color="black" if is_blend else None,
            linewidth=2.8 if is_blend else 1.2,
            linestyle="--" if is_blend else "-",
        )
    ax.set_title(metric.upper())
    ax.set_ylabel("GHI error [W/m²]")
    ax.grid(alpha=0.3)
    if metric == "bias":
        ax.axhline(0, color="grey", linestyle=":")
axes[-1].set_xlabel("Forecast lead time [hours]")
axes[0].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.suptitle("Selected blend and individual models on holdout dates")
plt.show()


## What we will conclude from the holdout

We should report whether the blend improves RMSE over the development leader in each lead interval, how many common pairs support that comparison, and whether bias becomes more or less systematic. Since the holdout is only a few days, we describe these findings as evidence for this archive and location, not as a universal ranking of forecast systems.


# 8. Measured and forecast GHI profiles with clear-sky index

The three dates were selected as examples of different measured daily profiles. For each date, we plot measured hourly GHI and the selected blend at a 24-hour lead. The right axis shows measured clear-sky index (K_c) for the same day. This connects forecast errors to the conditions observed at the station.

The profile labels are provisional descriptions. We use the (K_c) summaries above to check them, and we do not use the realized (K_c) to choose a forecast that would have been available in advance.


In [ ]:
profile_forecast = final_blend.get(PROFILE_LEAD_H)
if profile_forecast is None:
    raise ValueError(f"No selected blend is available at lead {PROFILE_LEAD_H} h")

for date, label in PROFILE_DAYS.items():
    start = pd.Timestamp(date, tz="UTC")
    end = start + pd.Timedelta(days=1)
    day = hourly_dmi.loc[
        (hourly_dmi.index > start) & (hourly_dmi.index <= end)
    ]
    forecast_day = profile_forecast.reindex(day.index)

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(day.index, day["ghi_measured"], marker="o", label="Measured GHI")
    ax.plot(
        day.index, forecast_day,
        marker="o", linestyle="--",
        label=f"Selected blend ({PROFILE_LEAD_H} h lead)",
    )
    ax.set_title(f"{date} — {label}: measured and forecast GHI")
    ax.set_xlabel("Valid time [UTC]")
    ax.set_ylabel("GHI [W/m²]")
    ax.grid(alpha=0.3)

    ax_kc = ax.twinx()
    ax_kc.plot(
        day.index, day["kc"],
        color="darkgreen", marker=".", alpha=0.75,
        label="Measured clear-sky index Kc",
    )
    ax_kc.axhline(1, color="darkgreen", linestyle=":", linewidth=1)
    ax_kc.set_ylabel("Clear-sky index Kc [-]", color="darkgreen")

    lines, labels = ax.get_legend_handles_labels()
    lines_kc, labels_kc = ax_kc.get_legend_handles_labels()
    ax.legend(lines + lines_kc, labels + labels_kc, loc="upper left")
    fig.tight_layout()
    plt.show()


# 9. Prepare the competition forecast

This section is intended to be run once the competition's forecast issue is available, by the submission deadline. The default issue time below is chosen so the requested 10–105-hour lead-time range covers 28 November 00:00 UTC through 1 December 23:00 UTC. Confirm the competition's exact period, time zone, and forecast cutoff before submission; update these three timestamps if needed.

Unlike the daylight skill analysis, the submitted file retains every required target hour. If the official score includes nighttime hours, our blend-selection objective should also be checked against that exact scoring rule.


In [ ]:
COMPETITION_ISSUE_TIME = pd.Timestamp("2026-11-27 14:00", tz="UTC")
COMPETITION_START = pd.Timestamp("2026-11-28 00:00", tz="UTC")
COMPETITION_END = pd.Timestamp("2026-12-01 23:00", tz="UTC")
COMPETITION_OUTPUT = project_root / "data" / "competition_blended_forecast.csv"

competition_leads = (
    pd.date_range(COMPETITION_START, COMPETITION_END, freq="1h")
    - COMPETITION_ISSUE_TIME
).total_seconds() / 3600

print(
    "Required lead-time range:",
    f"{competition_leads.min():.0f}-{competition_leads.max():.0f} hours",
)
if competition_leads.min() < 10 or competition_leads.max() > 105:
    print("Update the issue time or target dates: they do not fit the evaluated 10-105 h range.")


In [ ]:
# Run this cell on/after the chosen issue time, once its forecast files exist.
if pd.Timestamp.now(tz="UTC") < COMPETITION_ISSUE_TIME:
    print("The selected issue time is in the future. Rerun this section on the submission date.")
else:
    competition_raw = get_forecast_data(
        hourly_dmi,
        station=STATION,
        start=COMPETITION_ISSUE_TIME,
        end=COMPETITION_ISSUE_TIME,
        models=FORECAST_MODELS,
    )

    competition_times = pd.date_range(
        COMPETITION_START, COMPETITION_END, freq="1h"
    )
    competition_predictions = {}
    for model, frame in competition_raw.items():
        issued = frame.loc[
            (frame["issue_time"] == COMPETITION_ISSUE_TIME)
            & frame["valid_time"].isin(competition_times),
            ["valid_time", "ghi_forecast"],
        ].drop_duplicates("valid_time").set_index("valid_time")
        competition_predictions[model] = issued["ghi_forecast"].reindex(competition_times)

    forecast_matrix = pd.DataFrame(competition_predictions, index=competition_times)
    output_values = []
    for valid_time in competition_times:
        lead = (valid_time - COMPETITION_ISSUE_TIME).total_seconds() / 3600
        interval_weights = None
        for first, last, weights in final_weight_schedule:
            if first <= lead <= last:
                interval_weights = weights
                break

        if interval_weights is None:
            output_values.append(np.nan)
            continue

        model_values = forecast_matrix.loc[valid_time]
        if all(
            model in model_values.index and pd.notna(model_values[model])
            for model in interval_weights
        ):
            output_values.append(sum(
                weight * model_values[model]
                for model, weight in interval_weights.items()
            ))
        else:
            output_values.append(np.nan)

    competition_submission = pd.DataFrame({
        "ghi_forecast": output_values,
        "lead_time_h": competition_leads,
    }, index=competition_times)
    competition_submission.index.name = "valid_time_utc"

    missing_submission = competition_submission["ghi_forecast"].isna()
    print("Target hours:", len(competition_submission))
    print("Hours without a complete blend:", int(missing_submission.sum()))
    display(competition_submission.head())

    if missing_submission.any():
        display(competition_submission.loc[missing_submission])
        print("Resolve missing model forecasts before exporting the submission file.")
    else:
        competition_submission.to_csv(COMPETITION_OUTPUT)
        print(f"Saved competition forecast to: {COMPETITION_OUTPUT}")


# 10. Final decision notes

Before submitting, we record:

- Which model led in development for each lead interval?
- In which intervals did the equal-weight blend improve validation RMSE on common pairs?
- Did that improvement persist on the untouched holdout period?
- What were the bias direction and sample count?
- Does the selected issue time cover every required competition timestamp and lead time?
- Does the exported file use the competition's exact UTC/local-time convention and scoring rule?

Our final choice should be the simplest schedule that performed at least as well as the best single model on validation and remained credible on holdout. If the short archive does not support a stable gain, a strong single model is a defensible result; blending is not automatically better.
